# Outliers imputation — Mean method

**Method**: mean ± k× SD (classic deviation from mean).

**Input**: 
* **routine DHIS2** data (formatted and aligned)
    * from Dataset "**snt-dhis2-formatted**", `XXX_routine.parquet`

**Output**: 
All outputs saved to Dataset **snt-dhis2-outliers-imputation**, in the standard structure shared by all outliers pipelines (formatters in `code/snt_utils.r`):
* **Detection table**: `XXX_routine_outliers_detected.parquet`, long format, one row per facility × period × indicator. Columns: `PERIOD`, `YEAR`, `MONTH` (integer), `DATE`, `ADM1_NAME`, `ADM1_ID`, `ADM2_NAME`, `ADM2_ID`, `OU_ID`, `OU_NAME`, `INDICATOR` (character), `VALUE` (double), `OUTLIER_DETECTED` (logical), `OUTLIER_METHOD` (`"MEAN"`).
* **Imputed / Removed**: `XXX_routine_outliers_imputed.parquet`, `XXX_routine_outliers_removed.parquet`, wide routine format, one row per facility × period of the routine data. Columns: `PERIOD`, `YEAR`, `MONTH` (integer), `ADM1_NAME`, `ADM1_ID`, `ADM2_NAME`, `ADM2_ID`, `OU_ID`, `OU_NAME` (character), then one double column per indicator. Imputed: only flagged values replaced. Removed: flagged values set to `NA`, rows kept.
* 🐘 **Table** in ws **Database** for 📊 Shiny App: SNT Outliers Explorer

---------------------

In [ ]:
# Parameters (injected by pipeline: ROOT_PATH, DEVIATION_MEAN)
# ROOT_PATH <- "~/workspace"
# DEVIATION_MEAN <- 3

## 1. Setup

In [ ]:
# Pipeline path
PIPELINE_PATH <- file.path("~/workspace", "pipelines", "snt_dhis2_outliers_imputation_mean")

# Shared helpers for this pipeline (code)
source(file.path(PIPELINE_PATH, "utils", "snt_dhis2_outliers_imputation_mean.r"))
snt_paths <- init_snt_workspace(
  snt_pipeline_name = "snt_dhis2_outliers_imputation_mean",
  packages = c("data.table", "arrow", "tidyverse", "jsonlite", "DBI", "RPostgres")
)

# set output path
OUTPUT_DIR <- file.path(snt_paths$DATA_PATH, "dhis2", "outliers_imputation")

### 1.1. Validate parameters

In [ ]:
if (!exists("DEVIATION_MEAN")) DEVIATION_MEAN <- 3

### 1.2. Load and check `SNT_config` file

In [ ]:
# Load SNT config
config_json <- load_snt_config(file.path(snt_paths$CONFIG_PATH, 'SNT_config.json'))

# Configuration validation is handled in pipeline.py
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)
DHIS2_INDICATORS <- names(config_json$DHIS2_DATA_DEFINITIONS$DHIS2_INDICATOR_DEFINITIONS)
fixed_cols = c('PERIOD', 'YEAR', 'MONTH', 'ADM1_ID', 'ADM2_ID', 'OU_ID')

## 2. Load Data

### 2.1. **Routine** data (DHIS2) 

Formatted & aggregated data stored in OpenHEXA Dataset "**SNT_DHIS2_FORMATTED**"

In [ ]:
# Load file from dataset (formatting)
dhis2_routine <- load_routine_data(
  dataset_name = config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED,
  country_code = COUNTRY_CODE,
  required_indicators = DHIS2_INDICATORS
)

print(dim(dhis2_routine))
head(dhis2_routine, 2)

In [ ]:
# YEAR/MONTH casting is handled inside load_routine_data().

🔍 **Assert indicators are present**

In [ ]:
# Indicator validation is handled inside load_routine_data().

## 3. Outliers Detection

### 3.1. Transform routine data  

* **Pivot longer***: cols become rows

In [ ]:
dhis2_routine_long <- dhis2_routine %>%
    select(all_of(c(fixed_cols, DHIS2_INDICATORS))) %>%
    pivot_longer(cols = all_of(DHIS2_INDICATORS), names_to = "INDICATOR", values_to = "VALUE")

print(dim(dhis2_routine_long))
head(dhis2_routine_long, 2)

🔍 **Remove duplicated values**

In [ ]:
# check if there are any duplicates
duplicated <- dhis2_routine_long %>%
  group_by(ADM1_ID, ADM2_ID, OU_ID, PERIOD, YEAR, MONTH, INDICATOR) %>%
  summarise(n = dplyr::n(), .groups= "drop") %>%
  filter(n > 1L)

# Remove dups
if (nrow(duplicated) > 0) {
    log_msg(glue("Removing {nrow(duplicated)} duplicated values."))
    dhis2_routine_long <- dhis2_routine_long %>%
        distinct(ADM1_ID, ADM2_ID, OU_ID, PERIOD, YEAR, MONTH, INDICATOR, .keep_all = TRUE)
    head(duplicated)
}

### 3.2. Calculate **summary stats**
At `OU_ID` (Health Facility) x `INDICATOR`, calculate:
* mean
* median
* SD
* MAD
* Q1 (25th)
* Q3 (75th).

In [ ]:
# stats
dhis2_routine_stats <- dhis2_routine_long %>%
    group_by(across(all_of(c("ADM1_ID", "ADM2_ID", "OU_ID", "INDICATOR")))) %>%  # , YEAR?
    # group_by(across(all_of(c("ADM1_ID", "ADM2_ID", "YEAR", "INDICATOR")))) %>%  # in BDI it's by YEAR, instead of by FOSA
    mutate(
        # n = n(), # added for inspection  
        # n_positive = length(na.omit(VALUE)), # ⚠️ 2025-08-26: added for inspection  
        mean = ceiling(mean(VALUE, na.rm = TRUE)),
        median = ceiling(median(VALUE, na.rm = TRUE)),
        sd = ceiling(sd(VALUE, na.rm = TRUE)),
        mad = ceiling(mad(VALUE, constant = 1, na.rm = TRUE)), # 🚨 scale factor: `constant = 1` (default `constant = 1.4826`) 
        q1 = ceiling(quantile(VALUE, 0.25, na.rm = TRUE)), 
        q3 = ceiling(quantile(VALUE, 0.75, na.rm = TRUE))
      ) %>% 
      ungroup() 

dim(dhis2_routine_stats)
head(dhis2_routine_stats, 2)

### 3.3. Flag outlier values: **Mean** method (mean ± k× SD)

In [ ]:
# Outliers detection: Mean method only
dhis2_routine_outliers <- dhis2_routine_stats %>% 
    mutate(
        mean_lower_bound = mean - DEVIATION_MEAN * sd, 
        mean_upper_bound = mean + DEVIATION_MEAN * sd,
        !!sym(glue("OUTLIER_MEAN{DEVIATION_MEAN}SD")) := if_else(
          VALUE < mean_lower_bound | VALUE > mean_upper_bound,
          TRUE,
          FALSE
        ))

outlier_cols <- dhis2_routine_outliers %>% select(starts_with("OUTLIER_")) %>% names()
log_msg(paste0("Calculated column : ", paste(outlier_cols, collapse=", ")))

print(dim(dhis2_routine_outliers))
head(dhis2_routine_outliers, 2)

### 3.4. Flag `NA`s as non-outliers: 

This overall eventually makes all `VALUE == 0` into not-outlier, because upstream all `VALUE == 0` where replaced with `NA` to be ignored by the summary stats that defined the bundaries for outliers (mean, median, mad, sd).

In [ ]:
dhis2_routine_outliers <- dhis2_routine_outliers %>%
  mutate(across(starts_with("OUTLIER_"), ~ if_else(is.na(.x), FALSE, .x)))

### 3.5. Select outliers column (Mean method)

In [ ]:
# Select outlier columns
dhis2_routine_outliers_selection <- dhis2_routine_outliers %>% 
    select(any_of(c(fixed_cols, "INDICATOR", "VALUE")), starts_with("OUTLIER_"))

print(dim(dhis2_routine_outliers_selection))
head(dhis2_routine_outliers_selection, 2) # <----------------------------- OUTLIERS TABLE

In [ ]:
# log detection results (Mean method)
outliers_col <- colnames(dhis2_routine_outliers_selection)[startsWith(colnames(dhis2_routine_outliers_selection), "OUTLIER_")][1]
nr_of_outliers <- nrow(dhis2_routine_outliers_selection[dhis2_routine_outliers_selection[[outliers_col]] == TRUE, ])
perc_outliers <- nr_of_outliers / nrow(dhis2_routine_outliers_selection) * 100
log_msg(glue("Mean ({DEVIATION_MEAN}*SD): {nr_of_outliers} outliers ({sprintf('%.3f', perc_outliers)} % of data points)."))

## 4. Routine data imputation (Mean method)

Generate imputed and removed versions using moving average.

### 4.1. Impute values to outliers

Compute moving average column ([-1, +1] points window) to be used as imputation value.

In [ ]:
# Outlier column for Mean method
mean_column <- grep("^OUTLIER_MEAN", colnames(dhis2_routine_outliers_selection), value = TRUE)[1]

In [ ]:
# Impute outliers (Mean method)
log_msg("Running imputation for outliers detected using Mean method.")
dhis2_routine_outliers_mean_imputed <- impute_outliers(dhis2_routine_outliers_selection, mean_column, stat="mean")

### 4.2. Format output tables (Mean)

The notebook computes the published values; the standard formatters in `code/snt_utils.r` only lay them out, with the same column order, types and rows for every outliers pipeline:

* **detected**: `format_outliers_detected_table()` from the Mean flag column, with `OUTLIER_METHOD = "MEAN"`.
* **imputed**: `format_outliers_imputed_table()` from `VALUE_IMPUTED` (section 4.1).
* **removed**: this cell computes `VALUE_REMOVED` (`VALUE` with flagged values set to `NA`, every row kept), then `format_outliers_removed_table()` lays it out.

Names are taken unchanged from the routine data. Duplicates are removed in section 3.1; the formatters do not check for them.

In [ ]:
# Mean removal: flagged values set to NA, rows kept (adds VALUE_REMOVED)
dhis2_routine_outliers_mean_removed <- dhis2_routine_outliers_selection %>%
    mutate(VALUE_REMOVED = if_else(.data[[mean_column]], NA_real_, as.numeric(VALUE)))
log_msg(glue("Removal: {sum(dhis2_routine_outliers_mean_removed[[mean_column]])} outlier values set to NA."))

outputs <- list(
    detected = format_outliers_detected_table(
        outliers_long = dhis2_routine_outliers_selection,
        outlier_col = mean_column,
        method = "MEAN",
        routine_df = dhis2_routine
    ),
    imputed = format_outliers_imputed_table(
        outliers_long = dhis2_routine_outliers_mean_imputed,
        indicators = DHIS2_INDICATORS,
        routine_df = dhis2_routine
    ),
    removed = format_outliers_removed_table(
        outliers_long = dhis2_routine_outliers_mean_removed,
        indicators = DHIS2_INDICATORS,
        routine_df = dhis2_routine
    )
)

## 5. Export Output tables

Export tables as `.parquet` files to `data/` folder.

In [ ]:
for (name in names(outputs)) {
    file_name <- paste0(COUNTRY_CODE, "_routine_outliers_", name, ".parquet")
    write_parquet(outputs[[name]], file.path(OUTPUT_DIR, file_name))
    log_msg(glue("Exported {file_name} ({nrow(outputs[[name]])} rows)"))
}

In [ ]:
log_msg(glue("Results saved under: {OUTPUT_DIR}"))    